In [12]:
# 패키지 설치 (최초 1회)
# !pip install open_clip_torch diffusers transformers accelerate safetensors pandas pillow scikit-learn tqdm

In [13]:
import os
import random
import time
import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import convnext_small, ConvNeXt_Small_Weights

import open_clip
from tqdm.auto import tqdm

In [14]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DEVICE: cuda
GPU: NVIDIA RTX A5000


In [15]:
# 경로 설정
CSV_PATH   = "/home/piai/다운로드/dataset_with_image_path.csv"
IMAGE_ROOT = "/home/piai/다운로드/generated_images"
SAVE_DIR   = "/home/piai/다운로드/saved_models_m4"
os.makedirs(SAVE_DIR, exist_ok=True)

# CSV 컬럼명
IMAGE_COL   = "filename"
EMOTION_COL = "emotion"
VISUAL_COL  = "visual"
SPACE_COL   = "space"

# 하이퍼파라미터
BATCH_SIZE   = 32
NUM_WORKERS  = 2
IMG_SIZE     = 224
EPOCHS       = 15
LR           = 1e-4
WEIGHT_DECAY = 1e-4

# loss 가중치
EMOTION_W = 1.0
VISUAL_W  = 1.0
SPACE_W   = 1.5

# backbone / CLIP
BACKBONE_OUT    = 768
CLIP_MODEL_NAME = "ViT-B-32"
CLIP_PRETRAINED = "openai"
CLIP_OUT_DIM    = 512

In [16]:
# image_path 컬럼을 로컬 경로로 수정 후 저장
# (CSV의 image_path가 Google Drive 등 다른 경로일 경우 filename 기준으로 재설정)
df_fix = pd.read_csv(CSV_PATH)

def fix_image_path(row):
    fn = str(row.get("filename", ""))
    if fn and fn != "nan":
        return os.path.join(IMAGE_ROOT, fn)
    return ""

df_fix["image_path"] = df_fix.apply(fix_image_path, axis=1)
df_fix.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print(f"image_path 수정 완료: {len(df_fix)}행 → {CSV_PATH}")
print(df_fix[["filename", "image_path"]].head())

image_path 수정 완료: 4500행 → /home/piai/다운로드/dataset_with_image_path.csv
       filename                                     image_path
0  img_0000.png  /home/piai/다운로드/generated_images/img_0000.png
1  img_0001.png  /home/piai/다운로드/generated_images/img_0001.png
2  img_0002.png  /home/piai/다운로드/generated_images/img_0002.png
3  img_0003.png  /home/piai/다운로드/generated_images/img_0003.png
4  img_0004.png  /home/piai/다운로드/generated_images/img_0004.png


In [17]:
# CSV 통합 — 현재 단일 CSV 사용으로 불필요
# df_part1 = pd.read_csv("/path/to/part1.csv")
# df_part2 = pd.read_csv("/path/to/part2.csv")
# df_merged = pd.concat([df_part1, df_part2], ignore_index=True)
# df_merged.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
print("CSV 통합 스킵 (단일 CSV 사용)")

CSV 통합 스킵 (단일 CSV 사용)


In [18]:
# 이미지 통합 — 단일 폴더 사용으로 불필요
# import shutil
# for src_dir in ["/path/to/images_part1", "/path/to/images_part2"]:
#     for fname in os.listdir(src_dir):
#         shutil.copy(os.path.join(src_dir, fname), IMAGE_ROOT)
print("이미지 통합 스킵 (단일 폴더 사용)")

이미지 통합 스킵 (단일 폴더 사용)


In [19]:
df = pd.read_csv(CSV_PATH)
print(f"전체 CSV rows: {len(df)}")

# full_image_path 생성 (filename 기준)
def build_full_path(x):
    x = str(x)
    if os.path.isabs(x) and os.path.exists(x):
        return x
    return os.path.join(IMAGE_ROOT, os.path.basename(x))

df["full_image_path"] = df[IMAGE_COL].apply(build_full_path)

# 실제 존재하는 파일만 사용
df = df[df["full_image_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"실제 존재하는 이미지: {len(df)}개")
display(df.head())

전체 CSV rows: 4500
실제 존재하는 이미지: 3000개


,id,emotion,visual,space,tags,description,image_path,filename,full_image_path
0,1,cozy,cool,spacious,"cozy, cool, spacious",A spacious home party room with cool lighting ...,/home/piai/다운로드/generated_images/img_0000.png,img_0000.png,/home/piai/다운로드/generated_images/img_0000.png
1,2,relaxing,cool,private,"relaxing, cool, private",A cool-toned relaxing indoor party setting wit...,/home/piai/다운로드/generated_images/img_0001.png,img_0001.png,/home/piai/다운로드/generated_images/img_0001.png
2,3,lively,cool,private,"lively, cool, private",A lively home party interior with cool lightin...,/home/piai/다운로드/generated_images/img_0002.png,img_0002.png,/home/piai/다운로드/generated_images/img_0002.png
3,4,social,bright,casual,"social, bright, casual",A social home party scene featuring a casual d...,/home/piai/다운로드/generated_images/img_0003.png,img_0003.png,/home/piai/다운로드/generated_images/img_0003.png
4,5,social,cool,modern,"social, cool, modern",A modern home party room with cool lighting cr...,/home/piai/다운로드/generated_images/img_0004.png,img_0004.png,/home/piai/다운로드/generated_images/img_0004.png


In [20]:
emotion_le = LabelEncoder()
visual_le  = LabelEncoder()
space_le   = LabelEncoder()

df["emotion_idx"] = emotion_le.fit_transform(df[EMOTION_COL].astype(str))
df["visual_idx"]  = visual_le.fit_transform(df[VISUAL_COL].astype(str))
df["space_idx"]   = space_le.fit_transform(df[SPACE_COL].astype(str))

num_emotion = df["emotion_idx"].nunique()
num_visual  = df["visual_idx"].nunique()
num_space   = df["space_idx"].nunique()

print("num_emotion:", num_emotion, "|", list(emotion_le.classes_))
print("num_visual: ", num_visual,  "|", list(visual_le.classes_))
print("num_space:  ", num_space,   "|", list(space_le.classes_))

num_emotion: 5 | ['cozy', 'lively', 'relaxing', 'romantic', 'social']
num_visual:  4 | ['bright', 'cool', 'dark', 'warm']
num_space:   5 | ['casual', 'compact', 'modern', 'private', 'spacious']


In [21]:
df["stratify_key"] = (
    df["emotion_idx"].astype(str) + "_" +
    df["visual_idx"].astype(str) + "_" +
    df["space_idx"].astype(str)
)

key_counts = df["stratify_key"].value_counts()
rare_keys  = key_counts[key_counts < 2].index.tolist()
df["stratify_key_fixed"] = df["stratify_key"].copy()
df.loc[df["stratify_key"].isin(rare_keys), "stratify_key_fixed"] = "RARE"

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["stratify_key_fixed"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("train:", train_df.shape)
print("valid:", valid_df.shape)

train: (2400, 14)
valid: (600, 14)


In [22]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

valid_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# CLIP 전처리 (모델 내부 로딩과 별개로 preprocess만 추출)
_, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED
)
print("transforms ready")

/home/piai/anaconda3/envs/test_env/lib/python3.10/site-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


transforms ready


In [23]:
class MultiHeadImageDataset(Dataset):
    def __init__(self, df, image_transform, clip_transform):
        self.df = df.reset_index(drop=True)
        self.image_transform = image_transform
        self.clip_transform  = clip_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["full_image_path"]).convert("RGB")

        return {
            "image":      self.image_transform(image),
            "clip_image": self.clip_transform(image),
            "emotion":    torch.tensor(int(row["emotion_idx"]), dtype=torch.long),
            "visual":     torch.tensor(int(row["visual_idx"]),  dtype=torch.long),
            "space":      torch.tensor(int(row["space_idx"]),   dtype=torch.long),
            "path":       row["full_image_path"]
        }

In [24]:
train_ds = MultiHeadImageDataset(train_df, train_tf, clip_preprocess)
valid_ds = MultiHeadImageDataset(valid_df, valid_tf, clip_preprocess)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"train: {len(train_ds)}개 | valid: {len(valid_ds)}개")

train: 2400개 | valid: 600개


In [25]:
def make_class_weights(labels, num_classes):
    counts  = np.bincount(labels, minlength=num_classes)
    counts  = np.maximum(counts, 1)
    weights = len(labels) / (num_classes * counts)
    return torch.tensor(weights, dtype=torch.float32)

emotion_weights = make_class_weights(train_df["emotion_idx"].values, num_emotion).to(DEVICE)
visual_weights  = make_class_weights(train_df["visual_idx"].values,  num_visual).to(DEVICE)
space_weights   = make_class_weights(train_df["space_idx"].values,   num_space).to(DEVICE)

print("emotion_weights:", emotion_weights)
print("visual_weights: ", visual_weights)
print("space_weights:  ", space_weights)

emotion_weights: tensor([0.9959, 1.0063, 1.0300, 0.9816, 0.9877], device='cuda:0')
visual_weights:  tensor([1.0135, 0.9788, 0.9901, 1.0187], device='cuda:0')
space_weights:   tensor([1.0367, 0.9897, 1.0127, 0.9979, 0.9658], device='cuda:0')


In [26]:
class ConvNeXtCLIPMultiHead(nn.Module):
    def __init__(self, num_emotion, num_visual, num_space,
                 clip_model_name=CLIP_MODEL_NAME,
                 clip_pretrained=CLIP_PRETRAINED,
                 dropout=0.3):
        super().__init__()

        # ConvNeXt backbone
        backbone = convnext_small(weights=ConvNeXt_Small_Weights.IMAGENET1K_V1)
        self.backbone_features = backbone.features
        self.backbone_avgpool  = backbone.avgpool
        self.backbone_out_dim  = BACKBONE_OUT

        # CLIP image encoder (frozen)
        self.clip_model = open_clip.create_model(clip_model_name, pretrained=clip_pretrained)
        self.clip_model.eval()
        for p in self.clip_model.parameters():
            p.requires_grad = False
        self.clip_out_dim = CLIP_OUT_DIM

        # Fusion + heads
        fusion_dim = self.backbone_out_dim + self.clip_out_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.emotion_head = nn.Linear(512, num_emotion)
        self.visual_head  = nn.Linear(512, num_visual)
        self.space_head   = nn.Linear(512, num_space)

    def forward(self, image, clip_image):
        x = self.backbone_features(image)
        x = self.backbone_avgpool(x)
        x = torch.flatten(x, 1)

        with torch.no_grad():
            clip_feat = self.clip_model.encode_image(clip_image).float()

        x         = F.normalize(x, dim=1)
        clip_feat = F.normalize(clip_feat, dim=1)

        fused = self.fusion(torch.cat([x, clip_feat], dim=1))

        return self.emotion_head(fused), self.visual_head(fused), self.space_head(fused)

In [27]:
model = ConvNeXtCLIPMultiHead(
    num_emotion=num_emotion,
    num_visual=num_visual,
    num_space=num_space
).to(DEVICE)

criterion_emotion = nn.CrossEntropyLoss(weight=emotion_weights)
criterion_visual  = nn.CrossEntropyLoss(weight=visual_weights)
criterion_space   = nn.CrossEntropyLoss(weight=space_weights)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("model ready")

model ready


In [28]:
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    return acc, f1


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    emo_true, emo_pred = [], []
    vis_true, vis_pred = [], []
    spa_true, spa_pred = [], []

    for batch in tqdm(loader, desc="Train", leave=False):
        image      = batch["image"].to(DEVICE)
        clip_image = batch["clip_image"].to(DEVICE)
        emotion    = batch["emotion"].to(DEVICE)
        visual     = batch["visual"].to(DEVICE)
        space      = batch["space"].to(DEVICE)

        optimizer.zero_grad()
        emo_logits, vis_logits, spa_logits = model(image, clip_image)

        loss = (
            EMOTION_W * criterion_emotion(emo_logits, emotion) +
            VISUAL_W  * criterion_visual(vis_logits,  visual)  +
            SPACE_W   * criterion_space(spa_logits,   space)
        )
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        emo_true.extend(emotion.cpu().numpy()); emo_pred.extend(emo_logits.argmax(1).detach().cpu().numpy())
        vis_true.extend(visual.cpu().numpy());  vis_pred.extend(vis_logits.argmax(1).detach().cpu().numpy())
        spa_true.extend(space.cpu().numpy());   spa_pred.extend(spa_logits.argmax(1).detach().cpu().numpy())

    emo_acc, emo_f1 = compute_metrics(emo_true, emo_pred)
    vis_acc, vis_f1 = compute_metrics(vis_true, vis_pred)
    spa_acc, spa_f1 = compute_metrics(spa_true, spa_pred)

    return {
        "loss": total_loss / len(loader),
        "emotion_acc": emo_acc, "emotion_f1": emo_f1,
        "visual_acc":  vis_acc, "visual_f1":  vis_f1,
        "space_acc":   spa_acc, "space_f1":   spa_f1,
    }


@torch.no_grad()
def valid_one_epoch(model, loader):
    model.eval()
    total_loss = 0
    emo_true, emo_pred = [], []
    vis_true, vis_pred = [], []
    spa_true, spa_pred = [], []

    for batch in tqdm(loader, desc="Valid", leave=False):
        image      = batch["image"].to(DEVICE)
        clip_image = batch["clip_image"].to(DEVICE)
        emotion    = batch["emotion"].to(DEVICE)
        visual     = batch["visual"].to(DEVICE)
        space      = batch["space"].to(DEVICE)

        emo_logits, vis_logits, spa_logits = model(image, clip_image)

        loss = (
            EMOTION_W * criterion_emotion(emo_logits, emotion) +
            VISUAL_W  * criterion_visual(vis_logits,  visual)  +
            SPACE_W   * criterion_space(spa_logits,   space)
        )
        total_loss += loss.item()

        emo_true.extend(emotion.cpu().numpy()); emo_pred.extend(emo_logits.argmax(1).cpu().numpy())
        vis_true.extend(visual.cpu().numpy());  vis_pred.extend(vis_logits.argmax(1).cpu().numpy())
        spa_true.extend(space.cpu().numpy());   spa_pred.extend(spa_logits.argmax(1).cpu().numpy())

    emo_acc, emo_f1 = compute_metrics(emo_true, emo_pred)
    vis_acc, vis_f1 = compute_metrics(vis_true, vis_pred)
    spa_acc, spa_f1 = compute_metrics(spa_true, spa_pred)

    return {
        "loss": total_loss / len(loader),
        "emotion_acc": emo_acc, "emotion_f1": emo_f1,
        "visual_acc":  vis_acc, "visual_f1":  vis_f1,
        "space_acc":   spa_acc, "space_f1":   spa_f1,
        "emotion_true": emo_true, "emotion_pred": emo_pred,
        "visual_true":  vis_true, "visual_pred":  vis_pred,
        "space_true":   spa_true, "space_pred":   spa_pred,
    }

In [29]:
history  = []
best_score = -1
best_path  = os.path.join(SAVE_DIR, "best_m4_convnext_clip.pt")

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    train_result = train_one_epoch(model, train_loader, optimizer)
    valid_result = valid_one_epoch(model, valid_loader)
    scheduler.step()

    valid_score = (
        0.3 * valid_result["emotion_f1"] +
        0.3 * valid_result["visual_f1"]  +
        0.4 * valid_result["space_f1"]
    )

    row = {
        "epoch":      epoch,
        "train_loss": train_result["loss"],
        "valid_loss": valid_result["loss"],
        "emotion_acc": valid_result["emotion_acc"],
        "visual_acc":  valid_result["visual_acc"],
        "space_acc":   valid_result["space_acc"],
        "emotion_f1":  valid_result["emotion_f1"],
        "visual_f1":   valid_result["visual_f1"],
        "space_f1":    valid_result["space_f1"],
        "score":       valid_score
    }
    history.append(row)
    print(pd.DataFrame([row]).to_string(index=False))

    if valid_score > best_score:
        best_score = valid_score
        torch.save({
            "model_state_dict": model.state_dict(),
            "emotion_classes": list(emotion_le.classes_),
            "visual_classes":  list(visual_le.classes_),
            "space_classes":   list(space_le.classes_),
            "config": {
                "img_size":         IMG_SIZE,
                "clip_model_name":  CLIP_MODEL_NAME,
                "clip_pretrained":  CLIP_PRETRAINED
            }
        }, best_path)
        print(f"Best model saved: {best_path} | score={best_score:.4f}")


===== Epoch 1/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
     1    5.384522     5.30813     0.408333    0.443333   0.246667    0.371336   0.374713  0.181972 0.296604
Best model saved: /home/piai/다운로드/saved_models_m4/best_m4_convnext_clip.pt | score=0.2966

===== Epoch 2/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
     2    5.186744    5.018437     0.518333    0.448333       0.28    0.488132   0.417346  0.225335 0.361777
Best model saved: /home/piai/다운로드/saved_models_m4/best_m4_convnext_clip.pt | score=0.3618

===== Epoch 3/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
     3    4.877393    4.753712         0.57    0.488333   0.283333    0.560362   0.469374  0.238462 0.404306
Best model saved: /home/piai/다운로드/saved_models_m4/best_m4_convnext_clip.pt | score=0.4043

===== Epoch 4/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
     4     4.59934    4.562073          0.6       0.515   0.286667    0.596788   0.509069  0.244613 0.429602
Best model saved: /home/piai/다운로드/saved_models_m4/best_m4_convnext_clip.pt | score=0.4296

===== Epoch 5/15 =====


...

 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
    11    3.279619    4.321688     0.653333        0.54      0.315    0.654067   0.542204  0.310378 0.483033
Best model saved: /home/piai/다운로드/saved_models_m4/best_m4_convnext_clip.pt | score=0.4830

===== Epoch 12/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
    12    3.201758    4.319564     0.663333    0.528333   0.303333    0.663198   0.533363  0.297389 0.477924

===== Epoch 13/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
    13    3.148901    4.301614     0.656667        0.53      0.315    0.656651   0.533314  0.307191 0.479866

===== Epoch 14/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
    14    3.135818     4.31085     0.643333        0.53   0.311667    0.644833    0.53541   0.30429 0.475789

===== Epoch 15/15 =====


 epoch  train_loss  valid_loss  emotion_acc  visual_acc  space_acc  emotion_f1  visual_f1  space_f1    score
    15    3.125161    4.312319         0.66    0.536667   0.311667    0.659416   0.541582  0.306048 0.482718


In [30]:
history_df = pd.DataFrame(history)
display(history_df)

,epoch,train_loss,valid_loss,emotion_acc,visual_acc,space_acc,emotion_f1,visual_f1,space_f1,score
0,1,5.384522,5.308130,0.408333,0.443333,0.246667,0.371336,0.374713,0.181972,0.296604
1,2,5.186744,5.018437,0.518333,0.448333,0.280000,0.488132,0.417346,0.225335,0.361777
2,3,4.877393,4.753712,0.570000,0.488333,0.283333,0.560362,0.469374,0.238462,0.404306
3,4,4.599340,4.562073,0.600000,0.515000,0.286667,0.596788,0.509069,0.244613,0.429602
4,5,4.300737,4.417365,0.631667,0.535000,0.288333,0.621629,0.524294,0.283320,0.457105
5,6,4.063245,4.335948,0.651667,0.543333,0.308333,0.644949,0.537112,0.286781,0.469331
6,7,3.851554,4.368051,0.643333,0.536667,0.293333,0.640136,0.539162,0.290961,0.470174
7,8,3.656981,4.301480,0.671667,0.535000,0.293333,0.665752,0.538614,0.288227,0.476601
8,9,3.501516,4.319255,0.641667,0.531667,0.310000,0.638886,0.531162,0.305849,0.473354
9,10,3.372545,4.294883,0.663333,0.536667,0.310000,0.660024,0.537895,0.305234,0.481469


In [31]:
ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

final_valid = valid_one_epoch(model, valid_loader)

result_df = pd.DataFrame([{
    "model":       "ConvNeXt+CLIP_MultiHead (m4)",
    "emotion_acc": final_valid["emotion_acc"],
    "visual_acc":  final_valid["visual_acc"],
    "space_acc":   final_valid["space_acc"],
    "emotion_f1":  final_valid["emotion_f1"],
    "visual_f1":   final_valid["visual_f1"],
    "space_f1":    final_valid["space_f1"],
}])
display(result_df)

,model,emotion_acc,visual_acc,space_acc,emotion_f1,visual_f1,space_f1
0,ConvNeXt+CLIP_MultiHead (m4),0.653333,0.54,0.315,0.654067,0.542204,0.310378


In [32]:
print("=== Emotion Report ===")
print(classification_report(
    final_valid["emotion_true"], final_valid["emotion_pred"],
    target_names=emotion_le.classes_, zero_division=0
))

print("\n=== Visual Report ===")
print(classification_report(
    final_valid["visual_true"], final_valid["visual_pred"],
    target_names=visual_le.classes_, zero_division=0
))

print("\n=== Space Report ===")
print(classification_report(
    final_valid["space_true"], final_valid["space_pred"],
    target_names=space_le.classes_, zero_division=0
))

=== Emotion Report ===
              precision    recall  f1-score   support

        cozy       0.56      0.72      0.63       122
      lively       0.64      0.48      0.55       122
    relaxing       0.52      0.61      0.56       114
    romantic       0.81      0.76      0.78       120
      social       0.79      0.69      0.74       122

    accuracy                           0.65       600
   macro avg       0.67      0.65      0.65       600
weighted avg       0.67      0.65      0.65       600


=== Visual Report ===
              precision    recall  f1-score   support

      bright       0.54      0.47      0.50       147
        cool       0.47      0.45      0.46       152
        dark       0.75      0.64      0.69       152
        warm       0.46      0.60      0.52       149

    accuracy                           0.54       600
   macro avg       0.55      0.54      0.54       600
weighted avg       0.55      0.54      0.54       600


=== Space Report ===
        